# Figure 5A: COMPASS Metabolic Flux Analysis
## LC1 (Epithelial 0 / Lactocyte 0) vs LC2 (Epithelial 4 / Lactocyte 4)

Three panels produced:
- **(5A-i)** PCA — PC2 vs PC3, coloured by epithelial subcluster, 95% confidence ellipses for LC1/LC2
- **(5A-ii)** Dot-plot — Cohen's d by RECON2 subsystem
- **(5A-iii)** Volcano plots — Citric acid cycle & Fatty acid oxidation

Plus a **Supplemental Table** (Table S9) of all QC-filtered reactions.

Analysis is run twice: without donor FC018 (primary) and with FC018 (sensitivity check).

## 1. Imports & Setup

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.transforms as transforms
from matplotlib.patches import Ellipse
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import wilcoxon as scipy_wilcoxon
from adjustText import adjust_text

warnings.filterwarnings("ignore")
%matplotlib inline

# Illustrator-friendly PDF settings
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42

# Output directory
OUT = "."
print("Saving figures to current directory")

# Colour palettes
EPI_COLOURS = {
    "Epithelial 0": "#E24A33",
    "Epithelial 1": "#348ABD",
    "Epithelial 2": "#988ED5",
    "Epithelial 3": "#FBC15E",
    "Epithelial 4": "#8EBA42",
}
# Renamed for figure legend
EPI_LABELS = {
    "Epithelial 0": "Lactocyte 0",
    "Epithelial 1": "Lactocyte 1",
    "Epithelial 2": "Lactocyte 2",
    "Epithelial 3": "Lactocyte 3",
    "Epithelial 4": "Lactocyte 4",
}
LC1_COL = "#E1BE6A"
LC2_COL = "#40B0A6"
SIG_COL = "#542788"
NS_COL  = "#D3D3D3"
FIGSIZE = (5, 5)

print("\u2705 Imports complete")

## 2. Helper Functions

In [ ]:
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR correction."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranks = np.empty(n, dtype=int)
    ranks[order] = np.arange(1, n + 1)
    adj = pvals * n / ranks
    adj_mono = np.minimum.accumulate(adj[order][::-1])[::-1]
    result = np.empty(n)
    result[order] = adj_mono
    return np.minimum(result, 1.0)


def cohens_d_paired(x, y):
    """Paired Cohen's d: mean(diff) / SD(diff)."""
    diff = np.asarray(x) - np.asarray(y)
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-12)


def get_reaction_consistencies(penalties, min_range=1e-3):
    """Convert raw COMPASS penalties to activity scores."""
    df = -np.log(penalties + 1)
    df = df[df.max(axis=1) - df.min(axis=1) >= min_range]
    df = df - df.min().min()
    return df


def pseudobulk_paired_wilcoxon(consistencies, meta, lc1_subtypes, lc2_subtypes):
    """
    Donor-level pseudobulk paired Wilcoxon signed-rank test.
    For each reaction and each donor:
      LC1 score = mean flux across lc1_subtypes for that donor
      LC2 score = mean flux across lc2_subtypes for that donor
    """
    donors = sorted(meta["orig.ident"].unique())
    lc1_pb = pd.DataFrame(index=consistencies.index, columns=donors, dtype=float)
    lc2_pb = pd.DataFrame(index=consistencies.index, columns=donors, dtype=float)

    for donor in donors:
        lc1_idx = meta[(meta["orig.ident"] == donor) &
                       (meta["Celltype"].isin(lc1_subtypes))].index
        lc2_idx = meta[(meta["orig.ident"] == donor) &
                       (meta["Celltype"].isin(lc2_subtypes))].index
        if len(lc1_idx) > 0:
            lc1_pb[donor] = consistencies.loc[:, lc1_idx].mean(axis=1)
        if len(lc2_idx) > 0:
            lc2_pb[donor] = consistencies.loc[:, lc2_idx].mean(axis=1)

    valid_donors = [d for d in donors
                    if not lc1_pb[d].isna().all() and not lc2_pb[d].isna().all()]
    lc1_pb = lc1_pb[valid_donors]
    lc2_pb = lc2_pb[valid_donors]
    print(f"  Donors used: {valid_donors}  (n={len(valid_donors)})")

    stats, pvals, cds = [], [], []
    for rxn in consistencies.index:
        a = lc1_pb.loc[rxn].values.astype(float)
        b = lc2_pb.loc[rxn].values.astype(float)
        if np.all(a == b) or np.all(np.isnan(a - b)):
            stats.append(np.nan); pvals.append(1.0); cds.append(0.0)
            continue
        try:
            s, p = scipy_wilcoxon(a, b)
        except Exception:
            s, p = np.nan, 1.0
        stats.append(s); pvals.append(p); cds.append(cohens_d_paired(a, b))

    results = pd.DataFrame(
        {"wilcox_stat": stats, "wilcox_pval": pvals, "cohens_d": cds},
        index=consistencies.index
    )
    results["adjusted_pval"] = bh_fdr(results["wilcox_pval"].fillna(1).values)
    return results


def draw_confidence_ellipse(x, y, ax, n_std=1.96, **kwargs):
    """Draw a 95% confidence ellipse around (x, y) data."""
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=(np.mean(x), np.mean(y)), width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)


print("\u2705 Helper functions defined")
print("  LC1 = Epithelial 0 (Lactocyte 0)")
print("  LC2 = Epithelial 4 (Lactocyte 4)")

## 3. Curated Reaction Labels (for volcano plots)

In [ ]:
labeled_reactions = {
    # TCA cycle
    "SUCOASm_pos":     "succinate-CoA ligase",
    "SUCOASm_neg":     "succinate-CoA ligase",
    "SUCOAS1m_pos":    "succinate-CoA ligase",
    "SUCOAS1m_neg":    "succinate-CoA ligase",
    "SUCD1m_neg":      "succinate dehydrogenase",
    "r0509_pos":       "succinate dehydrogenase",
    "ICDHyrm_neg":     "isocitrate dehydrogenase",
    "MDHm_neg":        "malate dehydrogenase",
    "r0081_neg":       "alanine aminotransferase",
    "r0081_pos":       "alanine aminotransferase",
    # FAO
    "C181CPT1_pos":    "CPT1",
    "ETFQO_pos":       "ETFQO",
    "CSNAT2m_pos":     "carnitine acetyltransferase",
    "FACOAL180i_pos":  "FA-CoA ligase",
    "FACOAL226_pos":   "FA-CoA ligase",
    "FACOAL181i_pos":  "FA-CoA ligase",
    "FACOAL1821_neg":  "FA-CoA ligase",
    "FACOAL200_pos":   "FA-CoA ligase",
    "FACOAL1832_pos":  "FA-CoA ligase",
}
print(f"Curated labels: {len(labeled_reactions)} reactions")

## 4. Load & Preprocess Data

**Required input files** (place in same directory as this notebook):
- `reactions.tsv` — COMPASS raw penalty matrix (Supplemental Table S9)
- `Pseudo_df_metadata.tsv` — pseudobulk sample metadata with columns: orig.ident (donor ID), Celltype
- `reaction_metadata.csv` — RECON2 reaction annotations with columns: reaction_name, subsystem, EC_number, associated_genes, confidence, formula

To reproduce: place all three files in the working directory alongside this notebook.

In [ ]:
def load_and_preprocess(exclude_fc018=True):
    """
    Load reaction penalties, metadata, and reaction annotations.
    Returns: flux_df (reaction consistencies), meta, reaction_metadata
    """
    tag = "WITHOUT" if exclude_fc018 else "WITH"
    print(f"\n{'='*60}")
    print(f"  Loading data {tag} donor FC018")
    print(f"{'='*60}")

    # Reaction penalties
    rp = pd.read_csv("reactions.tsv", sep="\t", index_col=0)
    print(f"Raw reaction penalties: {rp.shape}")
    rp[rp <= 1e-4] = 0
    rp = rp[np.all(rp != 0, axis=1)]
    rp.columns = rp.columns.str.replace(r"^RNA\.", "", regex=True)
    if exclude_fc018:
        rp = rp.loc[:, ~rp.columns.str.contains("FC018")]
    rp = rp[rp.max(axis=1) - rp.min(axis=1) != 0]
    print(f"After QC filtering: {rp.shape}")

    # Metadata
    meta = pd.read_csv("Pseudo_df_metadata.tsv", sep="\t", index_col=0)
    if exclude_fc018:
        meta = meta[~meta["orig.ident"].str.contains("FC018")].copy()
    meta.index = meta.index.to_series().apply(
        lambda x: re.sub(r"(Epithelial)\s*(\d+)", r"\1.\2", x)
    )
    meta = meta.loc[rp.columns]
    print(f"Metadata: {meta.shape}")
    print(meta["Celltype"].value_counts().sort_index())

    # Reaction metadata
    rxn_meta = pd.read_csv("reaction_metadata.csv", index_col=0)

    # Consistency scores
    flux_df = get_reaction_consistencies(rp)
    print(f"Reaction consistency matrix: {flux_df.shape}")

    return flux_df, meta, rxn_meta


def run_differential(flux_df, meta, rxn_meta):
    """
    Run paired Wilcoxon on LC1 (Epi0) vs LC2 (Epi4), annotate, QC filter.
    Returns: W (annotated results DataFrame)
    """
    LC1_SUBTYPES = ["Epithelial 0"]
    LC2_SUBTYPES = ["Epithelial 4"]
    print(f"\nLC1 = {LC1_SUBTYPES}, LC2 = {LC2_SUBTYPES}")

    wilcox_results = pseudobulk_paired_wilcoxon(flux_df, meta, LC1_SUBTYPES, LC2_SUBTYPES)
    print(f"Reactions tested: {len(wilcox_results)}")

    # Annotate with reaction metadata
    wilcox_results["metadata_r_id"] = ""
    for r in wilcox_results.index:
        if r in rxn_meta.index:
            wilcox_results.loc[r, "metadata_r_id"] = r
        elif r[:-4] in rxn_meta.index:
            wilcox_results.loc[r, "metadata_r_id"] = r[:-4]

    W = wilcox_results.merge(
        rxn_meta, how="left",
        left_on="metadata_r_id", right_index=True, validate="m:1"
    )

    # Quality filters
    W = W[W["confidence"].isin([0, 4])]
    W = W[~W["EC_number"].isna()]

    # Reclassify cytoplasmic TCA reactions
    W.loc[
        W["formula"].map(lambda x: "[m]" not in str(x)) & (W["subsystem"] == "Citric acid cycle"),
        "subsystem"
    ] = "Other"

    n_sig_005 = (W["adjusted_pval"] < 0.05).sum()
    n_sig_01  = (W["adjusted_pval"] < 0.1).sum()
    print(f"Reactions after QC:        {len(W)}")
    print(f"Significant (FDR < 0.05):  {n_sig_005}")
    print(f"Significant (FDR < 0.1):   {n_sig_01}")

    return W

print("\u2705 Pipeline functions defined")

## 5. Run Analysis — WITHOUT FC019 (Primary)

In [ ]:
flux_df, meta, rxn_meta = load_and_preprocess(exclude_fc019=True)
W = run_differential(flux_df, meta, rxn_meta)

## 6. Figure A — PCA (PC2 vs PC3) without FC019

In [ ]:
X  = flux_df.T
Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=3, random_state=0)
pcs = pca.fit_transform(Xs)
pc_df = pd.DataFrame(pcs, index=X.index, columns=["PC1", "PC2", "PC3"]).join(meta)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
var3 = pca.explained_variance_ratio_[2] * 100
print(f"Variance explained: PC1={var1:.1f}%, PC2={var2:.1f}%, PC3={var3:.1f}%")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor("#FAFAFA")

for group, colour in EPI_COLOURS.items():
    grp = pc_df[pc_df["Celltype"] == group]
    ax.scatter(grp["PC2"], grp["PC3"],
               color=colour, s=90, alpha=0.90,
               edgecolors="white", linewidths=0.5,
               label=EPI_LABELS[group], zorder=3,
               rasterized=True)

# 95% confidence ellipses for Epi0 (LC1) and Epi4 (LC2)
epi0 = pc_df[pc_df["Celltype"] == "Epithelial 0"]
epi4 = pc_df[pc_df["Celltype"] == "Epithelial 4"]
draw_confidence_ellipse(epi0["PC2"].values, epi0["PC3"].values, ax,
                        facecolor=LC1_COL, alpha=0.15, edgecolor=LC1_COL,
                        linewidth=1.5, linestyle="--", zorder=2)
draw_confidence_ellipse(epi4["PC2"].values, epi4["PC3"].values, ax,
                        facecolor=LC2_COL, alpha=0.15, edgecolor=LC2_COL,
                        linewidth=1.5, linestyle="--", zorder=2)

# Ellipse legend entries
ax.plot([], [], color=LC1_COL, linestyle="--", linewidth=1.5, label="LC1-like (Lact 0)")
ax.plot([], [], color=LC2_COL, linestyle="--", linewidth=1.5, label="LC2-like (Lact 4)")

ax.axhline(0, color="lightgrey", lw=0.7, zorder=1)
ax.axvline(0, color="lightgrey", lw=0.7, zorder=1)
ax.set_xlabel(f"PC2 ({var2:.1f}%)", fontsize=12)
ax.set_ylabel(f"PC3 ({var3:.1f}%)", fontsize=12)
ax.set_title("COMPASS flux PCA — without FC019", fontsize=13, fontweight="bold", pad=10)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
          fontsize=9, frameon=True, framealpha=0.9, edgecolor="lightgrey")
plt.tight_layout()

path = os.path.join(OUT, "FigA_PCA_noFC019.pdf")
plt.savefig(path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved: {path}")
plt.show()
plt.close(fig)

## 7. Figure B — Dot-plot by Subsystem (without FC019)

In [ ]:
data_dot = W[~W['subsystem'].isin(["Miscellaneous", "Unassigned", "Other"])].copy()

# Keep subsystems with > 3 reactions
items, counts = np.unique(data_dot['subsystem'], return_counts=True)
valid_items = [items[i] for i in range(len(items)) if counts[i] > 3]
data_dot = data_dot[data_dot['subsystem'].isin(valid_items)]

# Sort: subsystems with sig reactions (FDR<0.01) by median |d|, then the rest
d_sig = data_dot[data_dot['adjusted_pval'] < 0.01].groupby('subsystem')['cohens_d'].median().abs()
sig_subsystems   = d_sig.sort_values(ascending=True).index.tolist()
other_subsystems = [s for s in data_dot.groupby('subsystem')['cohens_d'].median()
                    .sort_values().index if s not in sig_subsystems]
subsystem_order  = other_subsystems + sig_subsystems

print(f"Reactions plotted: {len(data_dot)}")
print(f"Subsystems shown:  {len(subsystem_order)}")

fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter([0] * len(subsystem_order), subsystem_order, alpha=0, s=0)

for subsystem in subsystem_order:
    subset = data_dot[data_dot['subsystem'] == subsystem]
    sig = subset['adjusted_pval'] < 0.05

    # Non-significant (faded)
    ax.scatter(
        subset.loc[~sig, 'cohens_d'], [subsystem] * (~sig).sum(),
        c=[LC1_COL if x >= 0 else LC2_COL for x in subset.loc[~sig, 'cohens_d']],
        s=30, alpha=0.18, edgecolor="none", zorder=2, rasterized=False
    )
    # Significant (filled)
    ax.scatter(
        subset.loc[sig, 'cohens_d'], [subsystem] * sig.sum(),
        c=[LC1_COL if x >= 0 else LC2_COL for x in subset.loc[sig, 'cohens_d']],
        s=40, alpha=1.0, edgecolor="none", zorder=3, rasterized=False
    )

for yi in range(len(subsystem_order) - 1):
    ax.axhline(yi + 0.5, color='#D3D3D3', lw=0.5, alpha=0.7, zorder=0)

ax.axvline(0, color='gray', linestyle='--', lw=1, zorder=1)
ax.set_xlabel("Cohen's d  (LC1 vs LC2, paired by donor)", fontsize=12)
ax.set_ylabel("Subsystem", fontsize=12)
ax.set_title("Effect sizes by subsystem (without FC019)\n"
             "(LC1 = gold, LC2 = teal; opaque = FDR < 0.05)", fontsize=13)
ax.tick_params(axis='y', labelsize=9)

legend_elements = [
    mpatches.Patch(facecolor=LC1_COL, label="Higher in LC1"),
    mpatches.Patch(facecolor=LC2_COL, label="Higher in LC2"),
    mpatches.Patch(facecolor="#D3D3D3", label="FDR \u2265 0.05 (faded)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9,
          frameon=True, framealpha=0.9)
plt.tight_layout()

path = os.path.join(OUT, "FigB_Dotplot_noFC019.pdf")
plt.savefig(path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved: {path}")
plt.show()
plt.close(fig)

## 8. Figure C — Volcano Plots: Citric Acid Cycle & Fatty Acid Oxidation (without FC019)

In [ ]:
def plot_volcano_panel(ax, data, title, labeled_reactions, fdr_thresh=0.1):
    """
    Volcano plot on a given axes. Only labels reactions in labeled_reactions dict.
    """
    data = data.copy().dropna(subset=["cohens_d", "adjusted_pval"])
    data["adjusted_pval"] = data["adjusted_pval"].clip(lower=1e-15)
    sig = data["adjusted_pval"] < fdr_thresh

    ymax = max(-np.log10(data["adjusted_pval"].min()) * 1.20, 2.5)

    # Non-significant
    ax.scatter(
        data.loc[~sig, "cohens_d"], -np.log10(data.loc[~sig, "adjusted_pval"]),
        c=NS_COL, s=45, alpha=0.5, edgecolor="none", zorder=1, rasterized=False
    )
    # Significant
    ax.scatter(
        data.loc[sig, "cohens_d"], -np.log10(data.loc[sig, "adjusted_pval"]),
        c=SIG_COL, s=55, alpha=0.95, edgecolor="none", zorder=2, rasterized=False
    )

    # Labels — ONLY curated reactions
    in_panel = data.index.intersection(labeled_reactions.keys())
    if len(in_panel) > 0:
        lbl_data = data.loc[in_panel]
        ax.scatter(
            lbl_data["cohens_d"], -np.log10(lbl_data["adjusted_pval"]),
            facecolors="none", edgecolors="black", s=90, linewidths=1.2, zorder=3
        )
        texts = []
        for rxn_id in in_panel:
            row = data.loc[rxn_id]
            x = row["cohens_d"]
            y = -np.log10(row["adjusted_pval"])
            enzyme_name = labeled_reactions[rxn_id]
            marker = " *" if row["adjusted_pval"] < fdr_thresh else ""
            txt = ax.text(x, y, f"{enzyme_name}{marker}",
                          fontsize=7, fontweight="bold", color="black", zorder=5)
            texts.append(txt)
        adjust_text(texts, ax=ax,
                    arrowprops=dict(arrowstyle="-", color="gray",
                                    lw=0.6, shrinkA=0, shrinkB=3),
                    expand=(1.3, 1.4),
                    force_text=(0.8, 1.0),
                    force_points=(0.3, 0.3))

    ax.set_ylim(0, ymax)
    ax.axvline(0, dashes=(3, 3), c="black", lw=1, zorder=0)
    ax.axhline(-np.log10(0.05), dashes=(3, 3), c="#666666", lw=1.0, zorder=0)
    ax.axhline(-np.log10(0.10), dashes=(4, 4), c="#AAAAAA", lw=1.0, zorder=0)
    ax.set_xlabel("Cohen's d  (\u2190 higher in LC2 | higher in LC1 \u2192)", fontsize=10)
    ax.set_ylabel("\u2212log\u2081\u2080(FDR)", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)

    # Count annotation
    ax.text(0.98, 0.98, f"n sig = {sig.sum()}/{len(data)}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8, color="grey")


# Build side-by-side volcano
tca_data = W[W["subsystem"] == "Citric acid cycle"]
fao_data = W[W["subsystem"] == "Fatty acid oxidation"]

print(f"Citric acid cycle: {len(tca_data)} reactions, "
      f"{(tca_data['adjusted_pval'] < 0.1).sum()} sig (FDR<0.1)")
print(f"Fatty acid oxidation: {len(fao_data)} reactions, "
      f"{(fao_data['adjusted_pval'] < 0.1).sum()} sig (FDR<0.1)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

plot_volcano_panel(ax1, tca_data, "Citric Acid Cycle", labeled_reactions)
plot_volcano_panel(ax2, fao_data, "Fatty Acid Oxidation", labeled_reactions)

plt.tight_layout()
path = os.path.join(OUT, "FigC_Volcanos_TCA_FAO_noFC019.pdf")
plt.savefig(path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved: {path}")
plt.show()
plt.close(fig)

## 9. Supplemental Table — All QC-filtered reactions (without FC019)

In [ ]:
# Build supplemental table from all QC-filtered reactions (the dotplot data)
supp = W[~W['subsystem'].isin(["Miscellaneous", "Unassigned", "Other"])].copy()
supp["direction"] = supp["cohens_d"].apply(lambda x: "Higher in LC1" if x >= 0 else "Higher in LC2")
supp["labeled_on_volcano"] = supp.index.isin(labeled_reactions.keys())
supp["significant_FDR005"] = supp["adjusted_pval"] < 0.05
supp["significant_FDR01"]  = supp["adjusted_pval"] < 0.1

# Select columns for output (including associated_genes)
supp_out = supp[[
    "reaction_name", "subsystem", "EC_number", "associated_genes",
    "cohens_d", "wilcox_pval", "adjusted_pval",
    "direction", "significant_FDR005", "significant_FDR01",
    "confidence", "formula"
]].copy()
supp_out.index.name = "reaction_id"
supp_out = supp_out.sort_values("adjusted_pval")

csv_path = os.path.join(OUT, "SuppTable_all_reactions_noFC019.csv")
supp_out.to_csv(csv_path)
print(f"Supplemental table saved: {csv_path}")
print(f"Total reactions: {len(supp_out)}")
print(f"Significant (FDR < 0.05): {supp_out['significant_FDR005'].sum()}")
print(f"Significant (FDR < 0.1):  {supp_out['significant_FDR01'].sum()}")
supp_out.head(15)

In [ ]:
supp.loc[supp.index.isin(labeled_reactions.keys())]

---
## 10. Sensitivity Analysis — WITH FC019

Re-run the full pipeline including donor FC019 to assess robustness.

In [ ]:
flux_df_fc019, meta_fc019, rxn_meta_fc019 = load_and_preprocess(exclude_fc019=False)
W_fc019 = run_differential(flux_df_fc019, meta_fc019, rxn_meta_fc019)

### Figure A — PCA with FC019

In [ ]:
X2  = flux_df_fc019.T
Xs2 = StandardScaler().fit_transform(X2)
pca2 = PCA(n_components=3, random_state=0)
pcs2 = pca2.fit_transform(Xs2)
pc_df2 = pd.DataFrame(pcs2, index=X2.index, columns=["PC1", "PC2", "PC3"]).join(meta_fc019)

var2_b = pca2.explained_variance_ratio_[1] * 100
var3_b = pca2.explained_variance_ratio_[2] * 100
print(f"Variance explained (with FC019): PC2={var2_b:.1f}%, PC3={var3_b:.1f}%")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_facecolor("#FAFAFA")
for group, colour in EPI_COLOURS.items():
    grp = pc_df2[pc_df2["Celltype"] == group]
    ax.scatter(grp["PC2"], grp["PC3"],
               color=colour, s=90, alpha=0.90,
               edgecolors="white", linewidths=0.5,
               label=EPI_LABELS[group], zorder=3, rasterized=True)

epi0_2 = pc_df2[pc_df2["Celltype"] == "Epithelial 0"]
epi4_2 = pc_df2[pc_df2["Celltype"] == "Epithelial 4"]
draw_confidence_ellipse(epi0_2["PC2"].values, epi0_2["PC3"].values, ax,
                        facecolor=LC1_COL, alpha=0.15, edgecolor=LC1_COL,
                        linewidth=1.5, linestyle="--", zorder=2)
draw_confidence_ellipse(epi4_2["PC2"].values, epi4_2["PC3"].values, ax,
                        facecolor=LC2_COL, alpha=0.15, edgecolor=LC2_COL,
                        linewidth=1.5, linestyle="--", zorder=2)
ax.plot([], [], color=LC1_COL, linestyle="--", linewidth=1.5, label="LC1-like (Lact 0)")
ax.plot([], [], color=LC2_COL, linestyle="--", linewidth=1.5, label="LC2-like (Lact 4)")

ax.axhline(0, color="lightgrey", lw=0.7, zorder=1)
ax.axvline(0, color="lightgrey", lw=0.7, zorder=1)
ax.set_xlabel(f"PC2 ({var2_b:.1f}%)", fontsize=12)
ax.set_ylabel(f"PC3 ({var3_b:.1f}%)", fontsize=12)
ax.set_title("COMPASS flux PCA — with FC019", fontsize=13, fontweight="bold", pad=10)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
          fontsize=9, frameon=True, framealpha=0.9, edgecolor="lightgrey")
plt.tight_layout()

path = os.path.join(OUT, "FigA_PCA_withFC019.pdf")
plt.savefig(path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved: {path}")
plt.show()
plt.close(fig)

### Figure B — Dot-plot with FC019

In [ ]:
data_dot2 = W_fc019[~W_fc019['subsystem'].isin(["Miscellaneous", "Unassigned", "Other"])].copy()
items2, counts2 = np.unique(data_dot2['subsystem'], return_counts=True)
valid_items2 = [items2[i] for i in range(len(items2)) if counts2[i] > 3]
data_dot2 = data_dot2[data_dot2['subsystem'].isin(valid_items2)]

d_sig2 = data_dot2[data_dot2['adjusted_pval'] < 0.01].groupby('subsystem')['cohens_d'].median().abs()
sig_sub2   = d_sig2.sort_values(ascending=True).index.tolist()
other_sub2 = [s for s in data_dot2.groupby('subsystem')['cohens_d'].median()
              .sort_values().index if s not in sig_sub2]
sub_order2 = other_sub2 + sig_sub2

fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter([0] * len(sub_order2), sub_order2, alpha=0, s=0)
for subsystem in sub_order2:
    subset = data_dot2[data_dot2['subsystem'] == subsystem]
    sig = subset['adjusted_pval'] < 0.05
    ax.scatter(subset.loc[~sig, 'cohens_d'], [subsystem] * (~sig).sum(),
               c=[LC1_COL if x >= 0 else LC2_COL for x in subset.loc[~sig, 'cohens_d']],
               s=30, alpha=0.18, edgecolor="none", zorder=2, rasterized=True)
    ax.scatter(subset.loc[sig, 'cohens_d'], [subsystem] * sig.sum(),
               c=[LC1_COL if x >= 0 else LC2_COL for x in subset.loc[sig, 'cohens_d']],
               s=40, alpha=1.0, edgecolor="none", zorder=3, rasterized=True)
for yi in range(len(sub_order2) - 1):
    ax.axhline(yi + 0.5, color='#D3D3D3', lw=0.5, alpha=0.7, zorder=0)
ax.axvline(0, color='gray', linestyle='--', lw=1, zorder=1)
ax.set_xlabel("Cohen's d  (LC1 vs LC2, paired by donor)", fontsize=12)
ax.set_ylabel("Subsystem", fontsize=12)
ax.set_title("Effect sizes by subsystem (with FC019)\n"
             "(LC1 = gold, LC2 = teal; opaque = FDR < 0.05)", fontsize=13)
ax.tick_params(axis='y', labelsize=9)
ax.legend(handles=[
    mpatches.Patch(facecolor=LC1_COL, label="Higher in LC1"),
    mpatches.Patch(facecolor=LC2_COL, label="Higher in LC2"),
    mpatches.Patch(facecolor="#D3D3D3", label="FDR \u2265 0.05 (faded)"),
], loc="lower right", fontsize=9, frameon=True, framealpha=0.9)
plt.tight_layout()
path = os.path.join(OUT, "FigB_Dotplot_withFC019.pdf")
plt.savefig(path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved: {path}")
plt.show()
plt.close(fig)

### Figure C — Volcano plots with FC019

In [ ]:
tca_data2 = W_fc019[W_fc019["subsystem"] == "Citric acid cycle"]
fao_data2 = W_fc019[W_fc019["subsystem"] == "Fatty acid oxidation"]

print(f"With FC019 — Citric acid cycle: {len(tca_data2)} rxn, "
      f"{(tca_data2['adjusted_pval'] < 0.1).sum()} sig")
print(f"With FC019 — Fatty acid oxidation: {len(fao_data2)} rxn, "
      f"{(fao_data2['adjusted_pval'] < 0.1).sum()} sig")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_volcano_panel(ax1, tca_data2, "Citric Acid Cycle", labeled_reactions)
plot_volcano_panel(ax2, fao_data2, "Fatty Acid Oxidation", labeled_reactions)
plt.tight_layout()
path = os.path.join(OUT, "FigC_Volcanos_TCA_FAO_withFC019.pdf")
plt.savefig(path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved: {path}")
plt.show()
plt.close(fig)

### Supplemental Table — with FC019

In [ ]:
supp2 = W_fc019[~W_fc019['subsystem'].isin(["Miscellaneous", "Unassigned", "Other"])].copy()
supp2["direction"] = supp2["cohens_d"].apply(lambda x: "Higher in LC1" if x >= 0 else "Higher in LC2")
supp2["significant_FDR005"] = supp2["adjusted_pval"] < 0.05
supp2["significant_FDR01"]  = supp2["adjusted_pval"] < 0.1

supp_out2 = supp2[[
    "reaction_name", "subsystem", "EC_number",
    "cohens_d", "wilcox_pval", "adjusted_pval",
    "direction", "significant_FDR005", "significant_FDR01",
    "confidence", "formula"
]].copy()
supp_out2.index.name = "reaction_id"
supp_out2 = supp_out2.sort_values("adjusted_pval")

csv_path2 = os.path.join(OUT, "SuppTable_all_reactions_withFC019.csv")
supp_out2.to_csv(csv_path2)
print(f"Supplemental table saved: {csv_path2}")
print(f"Total reactions: {len(supp_out2)}")
print(f"Significant (FDR < 0.05): {supp_out2['significant_FDR005'].sum()}")
print(f"Significant (FDR < 0.1):  {supp_out2['significant_FDR01'].sum()}")

## 11. Summary & Methods Update

In [ ]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print()
print("WITHOUT FC019 (primary):")
print(f"  Reactions after QC:        {len(W)}")
n01_nofc = (W['adjusted_pval'] < 0.1).sum()
n005_nofc = (W['adjusted_pval'] < 0.05).sum()
print(f"  Significant (FDR < 0.05):  {n005_nofc}")
print(f"  Significant (FDR < 0.1):   {n01_nofc}")
print()
print("WITH FC019 (sensitivity):")
print(f"  Reactions after QC:        {len(W_fc019)}")
n01_fc = (W_fc019['adjusted_pval'] < 0.1).sum()
n005_fc = (W_fc019['adjusted_pval'] < 0.05).sum()
print(f"  Significant (FDR < 0.05):  {n005_fc}")
print(f"  Significant (FDR < 0.1):   {n01_fc}")
print()
print("=" * 70)
print("METHODS TEXT UPDATE")
print("=" * 70)
print(f"\nUpdate the methods section with:")
print(f'  \"After quality filtering, {len(W)} reactions were retained, ')
print(f'   of which {n01_nofc} were significant at FDR < 0.1.\"')
print()
print("Also update LC groups to:")
print("  LC1 (Lactocyte 0) and LC2 (Lactocyte 4)")
print("  (not the multi-subtype groupings)")